# Meezan Branch Location Graph

Load the Meezan Excel file, create the branch hierarchy in Neo4j, and store branch coordinates as native Neo4j points.

## 1. Imports and configuration

In [20]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

PROJECT_ROOT = Path.cwd().parent
# xlsx_path = Path.cwd().parent / "data" / "raw" / "meezan.xlsx"
XLSX_PATH = PROJECT_ROOT / "data" / "raw" / "meezan.xlsx"

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

DATASET_ID = "meezan_branches"
BATCH_SIZE = 250

# Create the connection once and reuse graph.query() in every later cell.
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

print("Neo4j connection ready.")
print("Excel file:", XLSX_PATH)

Neo4j connection ready.
Excel file: /home/ekkel-ai/momina/projects/graphrag/data/raw/meezan.xlsx


## 2. Load the Excel file

In [21]:
df = pd.read_excel(
    XLSX_PATH,
    sheet_name="Sheet1",
    header=1,
    dtype=str,
)

df.columns = [str(column).strip() for column in df.columns]
df = df.apply(lambda column: column.str.strip() if column.dtype == "object" else column)

# Match the column names used in the Meezan workbook.
df = df.rename(columns={"City Name": "City", "Area Name": "Area"})

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Rows: 1102
Columns: ['S.No', 'Branch Code', 'Branch Name', 'OM Landline No', 'BM Landline No', 'Branch Landline Number', 'City', 'Area', 'Province', 'Branch Address', 'Latitude', 'Longitude']


,S.No,Branch Code,Branch Name,OM Landline No,BM Landline No,Branch Landline Number,City,Area,Province,Branch Address,Latitude,Longitude
0,1,105,"Marriott Hotel, Karachi",021-35693166,021-5682682,021-5683491,Karachi,Clifton,Sindh,"Marriott Hotel, Abdullah Haroon Road, Karachi",24.846098,67.0311
1,2,201,"Gulberg, Lahore",042-35755716\n(042-35777434),042-35870929,042-5879870-72,Lahore,Gulberg,Punjab,"60 Main Boulevard, Gulberg, Lahore",31.528028,74.350303
2,3,301,"Jinnah Avenue (Blue Area), Islamabad",051-2206513,051-2277518,051-2276712-15,Islamabad,Islamabad East,Islamabad,"Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ...",33.720747,73.076406
3,4,101,"PNSC (Lalazar), Karachi",021-35636253,021-35636252,021-35636404,Karachi,Chundrigar Road,Sindh,"37-A, Lalazar Area, Off Moulvi Tamizuddin Khan...",NaN,NaN
4,5,401,"Sereena Hotel, Faisalabad",041-2602594,041-2601225,041-2602595-97,Faisalabad,Commercial FSD,Punjab,"Shop No.G-737 to G-740, G-751 & G-753, Bazar C...",31.428856,73.082234


## 3. Prepare branch records

In [22]:
def clean_text(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value and value.lower() != "nan" else None


def clean_coordinate(value):
    value = clean_text(value)
    if value is None:
        return None
    try:
        return float(value)
    except ValueError:
        return None


required_columns = [
    "Branch Code", "Branch Name", "City", "Area", "Province",
    "Branch Address", "Latitude", "Longitude",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in Excel file: {missing_columns}")

records = []

for _, row in df.iterrows():
    branch_code = clean_text(row["Branch Code"])
    if not branch_code:
        continue

    # Meezan branch codes use four digits; preserve leading zeroes.
    branch_code = branch_code.split(".")[0].zfill(4)

    latitude = clean_coordinate(row["Latitude"])
    longitude = clean_coordinate(row["Longitude"])

    if latitude is not None and not -90 <= latitude <= 90:
        latitude = None
    if longitude is not None and not -180 <= longitude <= 180:
        longitude = None
    if latitude is None or longitude is None:
        latitude, longitude = None, None

    province = clean_text(row["Province"])
    city = clean_text(row["City"])
    area = clean_text(row["Area"])

    records.append({
        "dataset_id": DATASET_ID,
        "branch_code": branch_code,
        "branch_name": clean_text(row["Branch Name"]),
        "branch_address": clean_text(row["Branch Address"]),
        "city": city,
        "area": area,
        "province": province,
        "province_key": province.lower() if province else None,
        "city_key": f"{province.lower()}|{city.lower()}" if province and city else None,
        "area_key": f"{province.lower()}|{city.lower()}|{area.lower()}" if province and city and area else None,
        "latitude": latitude,
        "longitude": longitude,
    })

print("Branch records prepared:", len(records))
print("Records with coordinates:", sum(
    row["latitude"] is not None and row["longitude"] is not None
    for row in records
))

Branch records prepared: 1102
Records with coordinates: 525


## 4. Preview the prepared data

In [23]:
preview_df = pd.DataFrame(records)
display(preview_df.head(10))
print("Branches without coordinates:", preview_df["latitude"].isna().sum())

,dataset_id,branch_code,branch_name,branch_address,city,area,province,province_key,city_key,area_key,latitude,longitude
0,meezan_branches,0105,"Marriott Hotel, Karachi","Marriott Hotel, Abdullah Haroon Road, Karachi",Karachi,Clifton,Sindh,sindh,sindh|karachi,sindh|karachi|clifton,24.846098,67.031100
1,meezan_branches,0201,"Gulberg, Lahore","60 Main Boulevard, Gulberg, Lahore",Lahore,Gulberg,Punjab,punjab,punjab|lahore,punjab|lahore|gulberg,31.528028,74.350303
2,meezan_branches,0301,"Jinnah Avenue (Blue Area), Islamabad","Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ...",Islamabad,Islamabad East,Islamabad,islamabad,islamabad|islamabad,islamabad|islamabad|islamabad east,33.720747,73.076406
3,meezan_branches,0101,"PNSC (Lalazar), Karachi","37-A, Lalazar Area, Off Moulvi Tamizuddin Khan...",Karachi,Chundrigar Road,Sindh,sindh,sindh|karachi,sindh|karachi|chundrigar road,NaN,NaN
4,meezan_branches,0401,"Sereena Hotel, Faisalabad","Shop No.G-737 to G-740, G-751 & G-753, Bazar C...",Faisalabad,Commercial FSD,Punjab,punjab,punjab|faisalabad,punjab|faisalabad|commercial fsd,31.428856,73.082234
5,meezan_branches,0104,"Gulshan-e-Iqbal, Karachi","Shop No. 1, 2, 3 & 4, Plot No.B-41, Block 13-A...",Karachi,Gulshan-e-Iqbal,Sindh,sindh,sindh|karachi,sindh|karachi|gulshan-e-iqbal,24.906729,67.098657
6,meezan_branches,0402,"Kotwali Road, Faisalabad","P-66 Allama Iqbal Road (Kotwali Road), Faisalabad",Faisalabad,City Faisalabad,Punjab,punjab,punjab|faisalabad,punjab|faisalabad|city faisalabad,NaN,NaN
7,meezan_branches,0501,Nawan Shaher Branch Multan (Main Branch),"92,93,94, LMQ Road, Chowk Nawan Shaher, Multan",Multan,Ahmad Shah Abdali,Punjab,punjab,punjab|multan,punjab|multan|ahmad shah abdali,NaN,NaN
8,meezan_branches,0102,"Jodia Bazar, Karachi","H 9/A Darya Lal Street, Market Quarters Jodia ...",Karachi,Jodia Bazar,Sindh,sindh,sindh|karachi,sindh|karachi|jodia bazar,NaN,NaN
9,meezan_branches,0103,"F.T.C, Karachi","Ground Floor, Block-B, Finance & Trade Centre,...",Karachi,Shara-e-Faisal,Sindh,sindh,sindh|karachi,sindh|karachi|shara-e-faisal,24.859554,67.053578


Branches without coordinates: 577


## 5. Create constraints and indexes

These constraints keep branch codes unique within this dataset. Dataset-scoped keys prevent repeated city or area names in different provinces from being merged together.

In [25]:
schema_queries = [
    """CREATE CONSTRAINT branch_dataset_code_unique IF NOT EXISTS
       FOR (b:Branch) REQUIRE (b.dataset_id, b.branch_code) IS UNIQUE""",
    """CREATE CONSTRAINT province_dataset_key_unique IF NOT EXISTS
       FOR (p:Province) REQUIRE (p.dataset_id, p.name_key) IS UNIQUE""",
    """CREATE CONSTRAINT city_dataset_key_unique IF NOT EXISTS
       FOR (c:City) REQUIRE (c.dataset_id, c.name_key) IS UNIQUE""",
    """CREATE CONSTRAINT area_dataset_key_unique IF NOT EXISTS
       FOR (a:Area) REQUIRE (a.dataset_id, a.name_key) IS UNIQUE""",
]

for query in schema_queries:
    graph.query(query)

print("Constraints created")

Constraints created


## 6. Ingest branches into Neo4j

In [26]:
ingest_query = """
UNWIND $rows AS row

MERGE (p:Province {dataset_id: row.dataset_id, name_key: row.province_key})
SET p.name = row.province

MERGE (c:City {dataset_id: row.dataset_id, name_key: row.city_key})
SET c.name = row.city,
    c.province_key = row.province_key

MERGE (a:Area {dataset_id: row.dataset_id, name_key: row.area_key})
SET a.name = row.area,
    a.city_name = row.city,
    a.city_key = row.city_key,
    a.province_key = row.province_key

MERGE (b:Branch {dataset_id: row.dataset_id, branch_code: row.branch_code})
SET b.branch_name = row.branch_name,
    b.branch_address = row.branch_address,
    b.city = row.city,
    b.area = row.area,
    b.province = row.province,
    b.latitude = row.latitude,
    b.longitude = row.longitude,
    b.geocoding_status = CASE
        WHEN row.latitude IS NOT NULL AND row.longitude IS NOT NULL
        THEN 'coordinates_provided'
        ELSE 'coordinates_missing'
    END

MERGE (b)-[:LOCATED_IN]->(a)
MERGE (a)-[:WITHIN_CITY]->(c)
MERGE (c)-[:WITHIN_PROVINCE]->(p)
"""

for start in range(0, len(records), BATCH_SIZE):
    batch = records[start:start + BATCH_SIZE]
    graph.query(ingest_query, params={"rows": batch})
    print(f"Ingested {min(start + BATCH_SIZE, len(records))}/{len(records)}")

print("Branch graph ingestion complete.")

Ingested 250/1102
Ingested 500/1102
Ingested 750/1102
Ingested 1000/1102
Ingested 1102/1102
Branch graph ingestion complete.


## 7. Create native Neo4j spatial points

In [27]:
point_result = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
WHERE b.latitude IS NOT NULL AND b.longitude IS NOT NULL
SET b.location = point({latitude: b.latitude, longitude: b.longitude})
RETURN count(b) AS points_created
""", params={"dataset_id": DATASET_ID})

print("Spatial points created:", point_result[0]["points_created"])

graph.query("""
CREATE POINT INDEX branch_location_index IF NOT EXISTS
FOR (b:Branch)
ON (b.location)
""")
print("Point index created or already present.")

Spatial points created: 525
Point index created or already present.


## 8. Preview the graph

In [ ]:
counts = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
WITH count(b) AS branches
MATCH (a:Area {dataset_id: $dataset_id})
WITH branches, count(a) AS areas
MATCH (c:City {dataset_id: $dataset_id})
WITH branches, areas, count(c) AS cities
MATCH (p:Province {dataset_id: $dataset_id})
RETURN branches, areas, cities, count(p) AS provinces
""", params={"dataset_id": DATASET_ID})

display(pd.DataFrame(counts))

branch_preview = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
OPTIONAL MATCH (b)-[:LOCATED_IN]->(a:Area)
OPTIONAL MATCH (a)-[:WITHIN_CITY]->(c:City)
OPTIONAL MATCH (c)-[:WITHIN_PROVINCE]->(p:Province)
RETURN
    b.branch_code AS branch_code,
    b.branch_name AS branch_name,
    a.name AS area,
    c.name AS city,
    p.name AS province,
    b.latitude AS latitude,
    b.longitude AS longitude,
    b.location IS NOT NULL AS has_location
ORDER BY b.branch_code
LIMIT 15
""", params={"dataset_id": DATASET_ID})

display(pd.DataFrame(branch_preview))

## 9. Example: find nearby branches

In [28]:
# Replace these with the coordinates of the location to search from.
search_latitude = 31.5204
search_longitude = 74.3587
radius_m = 5000

nearby_branches = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
WHERE b.location IS NOT NULL
  AND point.distance(
      b.location,
      point({latitude: $latitude, longitude: $longitude})
  ) <= $radius_m
WITH b, point.distance(
    b.location,
    point({latitude: $latitude, longitude: $longitude})
) AS distance_m
RETURN
    b.branch_code AS branch_code,
    b.branch_name AS branch_name,
    b.city AS city,
    b.area AS area,
    round(distance_m / 1000.0, 2) AS distance_km
ORDER BY distance_m
LIMIT 10
""", params={
    "dataset_id": DATASET_ID,
    "latitude": search_latitude,
    "longitude": search_longitude,
    "radius_m": radius_m,
})

display(pd.DataFrame(nearby_branches))

,branch_code,branch_name,city,area,distance_km
0,1138,"Falcon Complex, Gulberg-III Lahore",Lahore,Gulberg,0.83
1,0245,"Ghalib Market, Lahore",Lahore,Gulberg,0.94
2,0290,"Liberty Market Branch, Lahore",Lahore,Gulberg,1.00
3,0226,"Punjab Coopeartive Housing Society (PCHS), Lahore",Lahore,Defence Lahore,1.15
4,0201,"Gulberg, Lahore",Lahore,Gulberg,1.16
5,0266,"Mini Market Gulberg II Branch, Lahore",Lahore,Gulberg,1.18
6,1160,"Tricon Corporate Center, Lahore",Lahore,Gulberg,1.18
7,0254,"Zahoor Elahi Road, Lahore (Ex: HSBC)",Lahore,Gulberg,1.45
8,0216,"Cavalry Ground, Lahore",Lahore,Gulberg,2.15
9,0294,"Jail Road Branch, Lahore\n(Ex: Zafar Ali Road ...",Lahore,Gulberg,2.15


In [30]:

import requests
import pandas as pd

# 1. Enter the location you want to search
search_location = "Anarkali, Lahore, Pakistan"

# 2. Geocode using your local Pakistan Nominatim server
NOMINATIM_URL = "http://localhost:8088/search"

response = requests.get(
    NOMINATIM_URL,
    params={
        "q": search_location,
        "format": "jsonv2",
        "limit": 1,
        "countrycodes": "pk",
        "addressdetails": 1,
    },
    headers={"User-Agent": "MeezanBranchSearch/1.0"},
    timeout=30,
)

response.raise_for_status()
results = response.json()

if not results:
    print("Location not found:", search_location)

else:
    result = results[0]

    latitude = float(result["lat"])
    longitude = float(result["lon"])

    print("Location:", result["display_name"])
    print("Latitude:", latitude)
    print("Longitude:", longitude)

    # 3. Find the five nearest Meezan branches in Neo4j
    nearest_branches = graph.query("""
        MATCH (b:Branch {dataset_id: $dataset_id})
        WHERE b.location IS NOT NULL

        WITH b,
             point.distance(
                 b.location,
                 point({
                     latitude: $latitude,
                     longitude: $longitude
                 })
             ) AS distance_m

        RETURN
            b.branch_code AS branch_code,
            b.branch_name AS branch_name,
            b.area AS area,
            b.city AS city,
            b.branch_address AS address,
            round(distance_m / 1000.0, 2) AS distance_km

        ORDER BY distance_m
        LIMIT 5
    """, params={
        "dataset_id": DATASET_ID,
        "latitude": latitude,
        "longitude": longitude,
    })

    # 4. Display results
    print("\nNearest Meezan branches:")
    display(pd.DataFrame(nearest_branches))


Location: انارکلی‎, لاہور, تحصیل لاہور شہر, ضلع لاہور, لاہور ڈویژن, پنجاب, 54100, پاکستان
Latitude: 31.5679445
Longitude: 74.3074319

Nearest Meezan branches:


,branch_code,branch_name,area,city,address,distance_km
0,0207,"Urdu Bazar, Lahore",City Lahore,Lahore,"4-kabeer street Urdu Bazar,Lahore",0.40
1,0272,"New Anarkali Bazar Branch, Lahore",City Lahore,Lahore,Property # S-64-R-132+134/2 New Anarkali Bazar...,0.69
2,0230,"Hall Road, Lahore",City Lahore,Lahore,"S-50-R-19 Hall Road, Lahore",1.15
3,0237,"Islampura, Lahore",Ravi,Lahore,"Property # 61 Main Bazar, Islampura, Lahore",1.19
4,0219,"Qartaba Chowk, Lahore",Allama Iqbal Town,Lahore,"Property # S-57-R/136/6 +A +B, Qartaba Chowk, ...",1.70
